# Library 03 – Checking the partnership data

A script that runs without errors is not proof that the data are right. This notebook defines three kinds
of checks:

1. **Structural checks** on the observation table (for example: the partnership runs of every innings add up
   to the runs of all its deliveries).
2. **Scorecard checks** that do not use the extractor at all. We add up each innings straight from the raw
   file and compare the result with facts Cricsheet records separately: the target set for the chasing side
   and the winning margin in runs.
3. **Spot checks** against well-known partnerships from public scorecards.

Needs: `lib_00_settings`.

In [ ]:
def raw_innings_totals(match_data: dict[str, Any]) -> list[dict[str, Any]]:
    """Innings totals computed directly from the raw deliveries (independent of the extractor)."""
    innings_totals = []
    for innings_index, innings_data in enumerate(match_data.get("innings", []), start=1):
        delivery_runs = sum(delivery["runs"]["total"]
                            for over in innings_data.get("overs", []) for delivery in over["deliveries"])
        penalty_runs = innings_data.get("penalty_runs", {}) or {}
        innings_totals.append({
            "innings_index": innings_index,
            "team": innings_data.get("team"),
            "delivery_runs": delivery_runs,
            "penalty": int(penalty_runs.get("pre", 0)) + int(penalty_runs.get("post", 0)),
            "super_over": bool(innings_data.get("super_over", False)),
            "target": (innings_data.get("target") or {}).get("runs"),
        })
    return innings_totals


def scorecard_checks(match_data: dict[str, Any], match_id: str, format_name: str) -> list[dict[str, Any]]:
    """Compare recomputed totals with the recorded target and run margin.

    Matches decided by a rain rule (``outcome.method``) are skipped, because their target does not follow
    from the first-innings total.
    """
    match_info = match_data["info"]
    match_outcome = match_info.get("outcome", {})
    normal_innings = [innings for innings in raw_innings_totals(match_data) if not innings["super_over"]]
    check_rows: list[dict[str, Any]] = []
    if not normal_innings or match_outcome.get("method"):
        return check_rows

    running_total = {team: 0 for team in match_info["teams"]}
    for position, innings in enumerate(normal_innings):
        # The target on the last innings should be: opponent's total - own earlier total + 1.
        is_last_innings = position == len(normal_innings) - 1
        if innings["target"] is not None and is_last_innings:
            other_team = next(team for team in match_info["teams"] if team != innings["team"])
            expected_target = running_total[other_team] - running_total[innings["team"]] + 1
            check_rows.append({"match_id": match_id, "format": format_name, "check": "target",
                               "recorded": int(innings["target"]), "recomputed": int(expected_target),
                               "passed": int(innings["target"]) == int(expected_target)})
        running_total[innings["team"]] += innings["delivery_runs"] + innings["penalty"]

    winning_margin = match_outcome.get("by", {})
    winning_team = match_outcome.get("winner")
    if winning_team and "runs" in winning_margin:
        losing_team = next(team for team in match_info["teams"] if team != winning_team)
        recomputed_margin = running_total[winning_team] - running_total[losing_team]
        check_rows.append({"match_id": match_id, "format": format_name, "check": "margin_runs",
                           "recorded": int(winning_margin["runs"]), "recomputed": int(recomputed_margin),
                           "passed": int(winning_margin["runs"]) == int(recomputed_margin)})
    return check_rows

## Structural checks on the observation table

Each check returns `(name, passed, detail)`.

In [ ]:
def observation_table_checks(observations: pd.DataFrame, innings_table: pd.DataFrame) -> list[tuple[str, bool, str]]:
    """Run every structural check and return the results."""
    check_results: list[tuple[str, bool, str]] = []

    def record(check_name: str, passed: bool, detail: str = "") -> None:
        check_results.append((check_name, bool(passed), detail))

    qualifying_innings = innings_table[innings_table["qualifying"]]
    record("innings_partnership_runs_equal_delivery_runs",
           (qualifying_innings["partnership_runs_sum"] == qualifying_innings["runs_from_deliveries"]).all(),
           f"{len(qualifying_innings)} qualifying innings, {int(qualifying_innings['runs_from_deliveries'].sum())} runs")
    record("distinct_players_in_pair", (observations["player_1_id"] != observations["player_2_id"]).all())
    record("canonical_pair_order", (observations["player_1_id"] < observations["player_2_id"]).all())
    unresolved_ids = (observations["player_1_id"].str.startswith("unresolved")
                      | observations["player_2_id"].str.startswith("unresolved"))
    record("all_player_ids_resolved", not unresolved_ids.any())
    record("runs_non_negative_integers", (observations["partnership_runs"] >= 0).all()
           and pd.api.types.is_integer_dtype(observations["partnership_runs"]))
    record("runs_equal_batter_plus_extras",
           (observations["partnership_runs"] == observations["batter_runs"] + observations["extras_runs"]).all())
    record("every_observation_has_a_delivery", (observations["deliveries"] >= 1).all())

    by_innings = observations.groupby(["match_id", "innings_index"])
    record("sequence_numbers_contiguous",
           (by_innings["partnership_seq"].max() == by_innings.size()).all()
           and (by_innings["partnership_seq"].min() == 1).all())
    is_last_partnership = observations["partnership_seq"] == by_innings["partnership_seq"].transform("max")
    record("innings_end_only_on_last_observation",
           not (observations.loc[~is_last_partnership, "boundary_reason"] == "innings_end").any())
    record("observation_count_matches_innings_table",
           int(qualifying_innings["n_partnerships"].sum()) == len(observations), f"{len(observations)} observations")
    record("only_england_batting", (observations["batting_team"] == TEAM_NAME).all())
    record("formats_canonical", set(observations["format"]) <= set(FORMATS))
    record("dates_inside_window", (observations["match_date"] >= STUDY_START_DATE).all()
           and (observations["match_date"] <= STUDY_END_DATE).all())
    return check_results

## Spot checks against documented partnerships

Five partnerships that are widely reported in public scorecards. If the extractor is right, it must find
exactly these run totals.

In [ ]:
DOCUMENTED_PARTNERSHIPS = [
    # (format, first day of the match, name part 1, name part 2, documented runs)
    ("Test", "2020-08-21", "Crawley", "Buttler", 359),
    ("Test", "2022-07-01", "Root", "Bairstow", 269),
    ("Test", "2022-12-01", "Crawley", "Duckett", 233),
    ("Test", "2024-10-07", "Root", "Brook", 454),
    ("T20I", "2022-11-10", "Buttler", "Hales", 170),
]


def documented_partnership_checks(observations: pd.DataFrame) -> pd.DataFrame:
    """Look up each documented partnership and compare the runs."""
    check_rows = []
    for format_name, match_day_text, name_part_1, name_part_2, documented_runs in DOCUMENTED_PARTNERSHIPS:
        match_day = pd.Timestamp(match_day_text).date()
        same_match = observations[(observations["format"] == format_name) & (observations["match_date"] == match_day)]
        pair_names = same_match["player_1_name"] + "|" + same_match["player_2_name"]
        matching_rows = same_match[pair_names.str.contains(name_part_1) & pair_names.str.contains(name_part_2)]
        extracted_runs = int(matching_rows["partnership_runs"].max()) if len(matching_rows) else None
        check_rows.append({"format": format_name, "match_date": match_day, "pair": f"{name_part_1} / {name_part_2}",
                           "documented_runs": documented_runs, "extracted_runs": extracted_runs,
                           "passed": extracted_runs == documented_runs})
    return pd.DataFrame(check_rows)